# Intrinsic dimension of CLIP encoder components

Per-**component** intrinsic-dimension diagnostics for the PRS residual-stream
decomposition, over **all models** (ViT-B/16, ViT-B/32, ViT-L/14, ViT-H/14,
RN50, RN101), **both towers** (image + text), and **all datasets** in
`output_dir` (imagenet, CIFAR100, caltech, fairface, binary_waterbirds).

A *component* is a single attention-head output or a single MLP output at a given
layer. For each component we build its `[N, d]` point cloud over the dataset and
estimate:

| col | meaning |
|-----|---------|
| **L** | linear ID = # principal components for **99%** of the variance (PCA, on s²) |
| **N** | **TwoNN** nonlinear intrinsic dimension (Facco et al., 2017) |
| **Ratio** | `L / N` — curvature / nonlinearity of the component manifold |
| **EVR1** | explained-variance ratio of the 1st principal component |
| **PC80 / PC95** | # principal components for 80% / 95% of the variance |

Three **views** of the residual stream are available per component (parallel-head
aware — heads inside a layer share the same layer input):

* `indiv` — the component's own output
* `cum_before` — running residual stream just *before* this component is added
* `cum_after` — `cum_before` + this component

> Heavy compute lives in `utils/scripts/intrinsic_dim.py`; it caches one CSV per
> `(dataset, model, tower)` under `output_dir/.../intrinsic_dim/`. This notebook
> only **loads + renders** those CSVs, so re-running it is cheap.
> Rebuild the cache with: `python -m utils.scripts.intrinsic_dim`.


In [1]:
import os, glob
import numpy as np
import pandas as pd

INPUT_DIR = "output_dir/activations_and_datasets_idxs_69"
ID_DIR    = os.path.join(INPUT_DIR, "intrinsic_dim")

paths = sorted(glob.glob(os.path.join(ID_DIR, "*.csv")))
df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True) if paths else pd.DataFrame()
print(f"{len(paths)} cached (model,tower,dataset) tables loaded  |  {len(df):,} rows")
if len(df):
    print("models  :", sorted(df.model.unique()))
    print("towers  :", sorted(df.tower.unique()))
    print("datasets:", sorted(df.dataset.unique()))
    print("views   :", sorted(df.view.unique()))


10 cached (model,tower,dataset) tables loaded  |  6,756 rows
models  : ['RN101', 'ViT-B-16', 'ViT-B-32']
towers  : ['image', 'text']
datasets: ['CIFAR100', 'imagenet']
views   : ['cum_after', 'cum_before', 'indiv']


## Rendering helpers

In [2]:
COLS = ["L", "N", "Ratio", "EVR1", "PC80", "PC95"]

def _label(r):
    return f"L{int(r.layer):02d}.h{int(r.comp):02d}" if r.kind == "attn" else f"L{int(r.layer):02d}.mlp"

def component_table(model, dataset, tower="image", view="indiv", kind=None):
    """Reference-style per-component table (one row per head / MLP)."""
    q = df[(df.model==model)&(df.dataset==dataset)&(df.tower==tower)&(df.view==view)]
    if kind: q = q[q.kind==kind]
    q = q.sort_values(["layer","kind","comp"]).copy()
    q.index = q.apply(_label, axis=1)
    t = q[COLS]
    return t.style.format({"L":"{:.0f}","N":"{:.2f}","Ratio":"{:.2f}",
                           "EVR1":"{:.2f}","PC80":"{:.0f}","PC95":"{:.0f}"})\
            .background_gradient(cmap="viridis", subset=["N"])\
            .set_caption(f"{model} · {tower} tower · {dataset} · view={view}"
                         + (f" · {kind}" if kind else ""))

def layer_summary(model, dataset, tower="image", view="indiv", kind="attn"):
    """Compact per-LAYER table: metrics averaged over the layer's heads
    (kind='attn') or the MLP row (kind='mlp'). Mirrors the classic ID-across-depth
    tables, but built from the per-component estimates."""
    q = df[(df.model==model)&(df.dataset==dataset)&(df.tower==tower)
           &(df.view==view)&(df.kind==kind)]
    g = q.groupby("layer")[COLS].mean()
    g.index.name = "Layer"
    return g.style.format({"L":"{:.2f}","N":"{:.2f}","Ratio":"{:.2f}",
                           "EVR1":"{:.2f}","PC80":"{:.2f}","PC95":"{:.2f}"})\
            .background_gradient(cmap="magma", subset=["N"])\
            .set_caption(f"{model} · {tower} · {dataset} · per-layer mean over {kind} · view={view}")


## 1. Per-component tables (headline)

One row per attention head / MLP. This is the finest-grained view — exactly the
`L N Ratio EVR1` table you asked for, at component granularity.

In [3]:
# pick the first available (model, dataset) as an example
if len(df):
    ex_model = "ViT-L-14" if "ViT-L-14" in df.model.unique() else sorted(df.model.unique())[0]
    ex_ds    = "imagenet" if "imagenet" in df[df.model==ex_model].dataset.unique() \
               else sorted(df[df.model==ex_model].dataset.unique())[0]
    print("example:", ex_model, ex_ds)
    display(component_table(ex_model, ex_ds, tower="image", view="indiv", kind="attn"))


example: RN101 CIFAR100


,L,N,Ratio,EVR1,PC80,PC95
L00.h00,57,18.20,3.13,0.28,17,42
L00.h01,55,17.50,3.14,0.36,12,35
L00.h02,58,18.83,3.08,0.18,21,44
L00.h03,57,18.53,3.08,0.16,19,42
L00.h04,59,20.94,2.82,0.24,21,45
L00.h05,58,19.95,2.91,0.24,19,44
L00.h06,59,19.43,3.04,0.25,21,46
L00.h07,42,12.01,3.50,0.25,6,19
L00.h08,59,19.81,2.98,0.20,21,46
L00.h09,58,19.05,3.04,0.27,19,44


In [4]:
# MLP components (image tower)
if len(df):
    display(component_table(ex_model, ex_ds, tower="image", view="indiv", kind="mlp"))


,L,N,Ratio,EVR1,PC80,PC95


## 2. Per-layer summary (compact, ID-across-depth)

Metrics averaged over each layer's attention heads — the compact form that
resembles the reference tables (rows = layers).

In [5]:
if len(df):
    display(layer_summary(ex_model, ex_ds, tower="image", view="indiv", kind="attn"))


,L,N,Ratio,EVR1,PC80,PC95
Layer,,,,,,
0,55.12,17.95,3.08,0.34,13.53,37.03
1,53.78,18.02,2.99,0.40,11.72,34.12
2,57.69,19.74,2.93,0.27,18.69,42.44
3,56.78,19.80,2.87,0.31,17.06,40.59
4,57.72,19.99,2.89,0.25,19.28,42.72
5,58.44,19.91,2.95,0.24,20.31,44.12
6,57.94,19.68,2.95,0.29,19.03,42.97
7,58.25,19.54,2.99,0.26,19.84,43.50
8,57.94,19.55,2.97,0.29,19.41,43.12


## 3. Cumulative residual-stream views

`cum_before` / `cum_after` show the intrinsic dimension of the *running* residual
stream around each component (parallel-head aware).

In [6]:
if len(df):
    display(component_table(ex_model, ex_ds, tower="image", view="cum_after", kind="attn"))


,L,N,Ratio,EVR1,PC80,PC95
L00.h00,34,14.77,2.30,0.90,1,5
L00.h01,34,13.59,2.50,0.85,1,8
L00.h02,34,15.71,2.16,0.91,1,6
L00.h03,30,15.61,1.92,0.92,1,5
L00.h04,36,16.57,2.17,0.91,1,6
L00.h05,32,15.44,2.07,0.92,1,4
L00.h06,35,16.63,2.11,0.92,1,5
L00.h07,31,10.70,2.90,0.62,4,11
L00.h08,29,16.41,1.77,0.94,1,3
L00.h09,43,15.42,2.79,0.85,1,12


## 4. Text tower

In [7]:
if len(df) and ((df.tower=="text").any()):
    tds = df[(df.model==ex_model)&(df.tower=="text")].dataset.unique()
    if len(tds):
        td = "imagenet" if "imagenet" in tds else sorted(tds)[0]
        display(component_table(ex_model, td, tower="text", view="indiv", kind="attn"))


,L,N,Ratio,EVR1,PC80,PC95
L00.h00,53,18.47,2.87,0.14,22,40
L00.h01,51,26.78,1.90,0.17,19,37
L00.h02,45,32.08,1.40,0.19,15,30
L00.h03,53,21.85,2.43,0.32,21,39
L00.h04,48,34.67,1.38,0.16,17,33
L00.h05,49,34.37,1.43,0.16,18,35
L00.h06,50,29.95,1.67,0.09,20,36
L00.h07,51,28.87,1.77,0.11,21,37
L01.h00,47,21.51,2.18,0.23,13,31
L01.h01,48,19.84,2.42,0.20,16,33


## 5. Comparison CSV — compare across models / datasets / towers

`export_comparison` writes two files under `intrinsic_dim/`:

* **`intrinsic_dim_all_components.csv`** — every component row, all models/towers/datasets (the full experiment data).
* **`intrinsic_dim_summary.csv`** — one *layer-aligned* row per `(model, tower, dataset, view, layer, kind)`; attn rows are the **mean over that layer's heads**, so they line up across models exactly like the reference tables.

Below we (re)build them and show side-by-side comparisons.

In [8]:
# pandas-only aggregation (mirrors utils.scripts.intrinsic_dim.export_comparison)
def export_comparison():
    all_df = df.copy()
    keys = ["model","tower","dataset","view","layer","kind"]
    summary = (all_df.groupby(keys, as_index=False)
                     .agg({**{m:"mean" for m in COLS}, "comp":"count"})
                     .rename(columns={"comp":"n_components"})
                     .sort_values(keys).reset_index(drop=True))
    all_df.to_csv(os.path.join(ID_DIR,"intrinsic_dim_all_components.csv"), index=False)
    summary.to_csv(os.path.join(ID_DIR,"intrinsic_dim_summary.csv"), index=False)
    print(f"wrote intrinsic_dim_all_components.csv ({len(all_df):,} rows)")
    print(f"wrote intrinsic_dim_summary.csv ({len(summary):,} rows)")
    return all_df, summary

all_df, summary = export_comparison()
summary.head()


wrote intrinsic_dim_all_components.csv (6,756 rows)
wrote intrinsic_dim_summary.csv (750 rows)


,model,tower,dataset,view,layer,kind,L,N,Ratio,EVR1,PC80,PC95,n_components
0,RN101,image,CIFAR100,cum_after,0,attn,25.0000,14.096696,1.727761,0.906488,1.09375,4.31250,32
1,RN101,image,CIFAR100,cum_after,1,attn,310.5000,19.019229,16.328096,0.269190,27.37500,147.93750,32
2,RN101,image,CIFAR100,cum_after,2,attn,254.0625,17.640111,14.403485,0.605198,8.31250,82.18750,32
3,RN101,image,CIFAR100,cum_after,3,attn,301.4375,19.990235,15.080207,0.508406,19.81250,134.25000,32
4,RN101,image,CIFAR100,cum_after,4,attn,333.5625,21.280981,15.676298,0.365273,38.00000,177.71875,32


In [9]:
def compare(metric="N", view="indiv", kind="attn", tower="image", dataset="imagenet"):
    """Wide layer x model table for one metric — the reference-style side-by-side."""
    q = summary[(summary.view==view)&(summary.kind==kind)
                &(summary.tower==tower)&(summary.dataset==dataset)]
    w = q.pivot_table(index="layer", columns="model", values=metric)
    return w.style.format("{:.2f}").background_gradient(cmap="viridis", axis=None)\
            .set_caption(f"{metric} — {tower} tower · {dataset} · {kind} · view={view} "
                         f"(attn = mean over heads)")

# TwoNN intrinsic dimension per layer, every model side by side
if len(summary):
    ds0 = "imagenet" if (summary.dataset=="imagenet").any() else summary.dataset.iloc[0]
    display(compare("N",     dataset=ds0))


model,ViT-B-16,ViT-B-32
layer,,
0,16.37,16.16
1,15.04,14.97
2,16.99,17.00
3,21.83,20.12
4,23.05,24.01
5,24.70,24.01
6,24.91,25.35
7,23.21,25.03
8,23.53,23.69


In [10]:
# same layout for the other metrics
if len(summary):
    display(compare("L",     dataset=ds0))   # 99%-variance linear dim
    display(compare("Ratio", dataset=ds0))   # L / N
    display(compare("EVR1",  dataset=ds0))   # PC1 explained-variance ratio


model,ViT-B-16,ViT-B-32
layer,,
0,39.83,33.50
1,39.00,37.67
2,44.42,44.67
3,56.92,54.00
4,58.67,59.42
5,60.33,59.58
6,61.08,61.17
7,61.08,61.67
8,61.67,61.75


model,ViT-B-16,ViT-B-32
layer,,
0,2.36,1.89
1,2.53,2.47
2,2.59,2.60
3,2.61,2.70
4,2.55,2.49
5,2.45,2.50
6,2.47,2.42
7,2.65,2.48
8,2.64,2.62


model,ViT-B-16,ViT-B-32
layer,,
0,0.53,0.72
1,0.42,0.47
2,0.25,0.29
3,0.18,0.26
4,0.17,0.15
5,0.14,0.16
6,0.13,0.15
7,0.13,0.11
8,0.18,0.13


## 6. Browse everything

Loop over every cached `(model, tower, dataset)` and show the per-component
individual-view table. Uncomment to render all (can be long).

In [11]:
def available():
    return (df[["model","tower","dataset"]].drop_duplicates()
              .sort_values(["model","tower","dataset"]).reset_index(drop=True)) if len(df) else df
available()


,model,tower,dataset
0,RN101,image,CIFAR100
1,RN101,text,CIFAR100
2,ViT-B-16,image,CIFAR100
3,ViT-B-16,image,imagenet
4,ViT-B-16,text,CIFAR100
5,ViT-B-16,text,imagenet
6,ViT-B-32,image,CIFAR100
7,ViT-B-32,image,imagenet
8,ViT-B-32,text,CIFAR100
9,ViT-B-32,text,imagenet


In [12]:
# for _, r in available().iterrows():
#     display(component_table(r.model, r.dataset, tower=r.tower, view="indiv"))
